In [1]:
import json
import pandas as pd
import re

In [2]:
def _extract_boxed_answer(response):
    # Find all \boxed{...} matches
    # This pattern will match nested braces within \boxed{} properly
    matches = re.findall(r'\\boxed{([^{}]*(?:\{[^{}]*\}[^{}]*)*)}', response)
    # If matches found, return the last one, otherwise return None
    if matches:
        return matches[-1]
    return None

In [ ]:
from datasets import load_dataset
from utils.MATH_500_eval.grader import grade_answer


dataset = load_dataset("OpenAI/gsm8k", "main", split='test')
df_dataset = pd.DataFrame(dataset)
df_dataset['id'] = range(len(df_dataset))  # 0-based index
df_dataset_meta = df_dataset[['id', 'answer']]

/local/scratch/bmg44/anaconda3/envs/r2a/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
TYPES = {
    "OG": "./output/reasoning_traces/gsm8k_DeepSeek-R1-Distill-Llama-8B_withoutR_finished_greedy_0.jsonl",
    "SOM": "./output/reasoning_traces/gsm8k_SOM_rephrased_DeepSeek-R1-Distill-Llama-8B_withoutR_finished_greedy_0.jsonl"
}

### Load datasets without reasoning (greedy)

In [5]:
for key in TYPES.keys():
    globals()[f"{key}_dataset_withoutR"] = [json.loads(line) for line in open(TYPES[key], "r").readlines()]
    print(globals()[f"{key}_dataset_withoutR"][0])
    globals()[f"df_{key}_dataset_withoutR"] = pd.DataFrame(globals()[f"{key}_dataset_withoutR"])
    globals()[f"df_{key}_dataset_withoutR"] =  globals()[f"df_{key}_dataset_withoutR"].rename(columns={'id': 'id', 'prompt': 'problem'})
    globals()[f"df_{key}_dataset_withoutR"]['response'] = globals()[f"df_{key}_dataset_withoutR"]['response'] + '<｜end▁of▁sentence｜>'
    globals()[f"df_{key}_dataset_withoutR"]['response'] = '<think>\n\n</think>' + globals()[f"df_{key}_dataset_withoutR"]['response'] 


    globals()[f"df_{key}_dataset_withoutR"]['is_response_finished'] = globals()[f"df_{key}_dataset_withoutR"]['response'].apply(lambda x: 1 if x.endswith('<｜end▁of▁sentence｜>') else 0)
    globals()[f"df_{key}_dataset_withoutR"]['answer_pred'] = globals()[f"df_{key}_dataset_withoutR"]['response'].apply(lambda x: x.split('</think>')[-1] if '</think>' in x else "")
    globals()[f"df_{key}_dataset_withoutR"]['extracted_answer'] = globals()[f"df_{key}_dataset_withoutR"]['answer_pred'].apply(_extract_boxed_answer)
    globals()[f"df_{key}_dataset_withoutR"]['is_answer_extracted'] = globals()[f"df_{key}_dataset_withoutR"]['extracted_answer'].apply(lambda x: 1 if x else 0)
    globals()[f"df_{key}_dataset_withoutR"]['end_think_count'] = globals()[f"df_{key}_dataset_withoutR"]['response'].apply(lambda x: len(x.split('</think>'))) # count the number of </think> tags
    print(globals()[f"df_{key}_dataset_withoutR"])


    df_merge = pd.merge(globals()[f"df_{key}_dataset_withoutR"], df_dataset_meta, how='left', left_on='unique_id', right_on='id')
    df_merge['is_answer_correct'] = df_merge.apply(
        lambda row: int(grade_answer(row['extracted_answer'], row['answer'].split("####")[-1].strip())),
        axis=1
    )
    df_merge.head(2)

    globals()[f"df_{key}_dataset_correct"] = df_merge[df_merge['is_answer_correct'].astype(bool)]
    globals()[f"df_{key}_dataset_incorrect"] = df_merge[~df_merge['is_answer_correct'].astype(bool)]

    print(globals()[f"df_{key}_dataset_correct"].head(2))
    print(globals()[f"df_{key}_dataset_incorrect"].head(2))

    

{'unique_id': 0, 'problem': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'response': "<｜begin▁of▁sentence｜>Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market? Please reason step by step, and put your final answer within \\boxed{}. <think>\nOkay, I think I have finished thinking.\n</think>\n\nLet's break down the problem step by step to find out how much Janet makes every day at the farmers' market.\n\n1. **Total Eggs Laid per Day:**\n   - Janet's ducks lay **16 eggs** each day.\n\n2. **Eggs Consumed:**\n   - **3 eggs** for bre

In [17]:
df_SOM_dataset_correct = globals()[f"df_SOM_dataset_correct"].rename(columns={'problem': 'parapharsed_problem', 'answer': 'gold_final_answer'})
df_SOM_dataset_incorrect = globals()[f"df_SOM_dataset_incorrect"].rename(columns={'problem': 'parapharsed_problem', 'answer': 'gold_final_answer'})
df_OG_dataset_correct = globals()[f"df_OG_dataset_correct"].rename(columns={'problem': 'original_problem','answer': 'gold_final_answer'})
df_OG_dataset_incorrect = globals()[f"df_OG_dataset_incorrect"].rename(columns={'problem': 'original_problem','answer': 'gold_final_answer'})

print(df_OG_dataset_correct.columns)

Index(['unique_id', 'original_problem', 'response', 'is_response_finished',
       'answer_pred', 'extracted_answer', 'is_answer_extracted',
       'end_think_count', 'id', 'gold_final_answer', 'is_answer_correct'],
      dtype='object')


In [35]:
df_SOM_correct_OG_incorrect = df_SOM_dataset_correct.merge(
    df_OG_dataset_incorrect[['unique_id', 'original_problem']],
    on='unique_id',
    how='inner'
).filter(items=['unique_id', 'original_problem', 'parapharsed_problem', 'gold_final_answer']).rename(columns={'unique_id': 'id'})
df_SOM_correct_OG_incorrect['paraphrase_type'] = "SOM"
df_SOM_correct_OG_incorrect[['gold_cot', 'gold_final_answer']] = (
    df_SOM_correct_OG_incorrect['gold_final_answer']
    .str.split("####", n=1, expand=True)
    .apply(lambda col: col.str.strip())
)
print(len(df_SOM_correct_OG_incorrect))
print(df_SOM_correct_OG_incorrect.columns)
df_SOM_correct_OG_incorrect.to_json("./data/rephrased/df_SOM_correct_OG_incorrect.jsonl", orient="records", lines=True)


116
Index(['id', 'original_problem', 'parapharsed_problem', 'gold_final_answer',
       'paraphrase_type', 'gold_cot'],
      dtype='object')


In [36]:
df_SOM_incorrect_OG_correct = df_SOM_dataset_incorrect.merge(
    df_OG_dataset_correct[['unique_id', 'original_problem']],
    on='unique_id',
    how='inner'
).filter(items=['unique_id', 'original_problem', 'parapharsed_problem', 'gold_final_answer']).rename(columns={'unique_id': 'id'})
df_SOM_incorrect_OG_correct['paraphrase_type'] = "SOM"
df_SOM_incorrect_OG_correct[['gold_cot', 'gold_final_answer']] = (
    df_SOM_incorrect_OG_correct['gold_final_answer']
    .str.split("####", n=1, expand=True)
    .apply(lambda col: col.str.strip())
)
print(len(df_SOM_incorrect_OG_correct))
print(df_SOM_incorrect_OG_correct.columns)
df_SOM_incorrect_OG_correct.to_json("./data/rephrased/df_SOM_incorrect_OG_correct.jsonl", orient="records", lines=True)


171
Index(['id', 'original_problem', 'parapharsed_problem', 'gold_final_answer',
       'paraphrase_type', 'gold_cot'],
      dtype='object')
